In [7]:
import pandas as pd
import json

# Alpet JSON dosyasını okuyalım
with open("../data/raw/benzinistasyonu/alpet.json", "r", encoding="utf-8") as f:
    alpet_data = json.load(f)

# Pandas DataFrame haline getirip kaç istasyon olduğuna bakalım
df_alpet_raw = pd.DataFrame(alpet_data)
print(f"Alpet toplam istasyon sayısı: {len(df_alpet_raw)}")

# Sütun isimlerini görebilmek için ilk 3 satırı ekrana basalım
df_alpet_raw.head(3)

Alpet toplam istasyon sayısı: 373


,SapCode,StationCode,AtmID,EpdkLicenceCode,Name,Status,StationType,ReplicationTime,IpAddress,Enlem,Boylam,Address,CityName,TownName,StationFeatures
0,100118,26004,270,BAY/939-82/24853,ESKİŞEHİR MERKEZ PETROL VE TİCARET ANONİM ŞİRKETİ,Aktif,İstasyonlu,/Date(1764709199000)/,192.168.20.2;192.168.1.128;192.168.250.251;20....,39.797451,30.521778,Esentepe Mahallesi No:360 Çevre Yolu ( Ada: ...,ESKİŞEHİR,ESKISEHIR,"[{'Key': 'AdBlueVar', 'Value': False}, {'Key':..."
1,100343,42069,1635,BAY/454-716/04026,BEYDAĞI PETROL MADENCİLİK TARIM ÜRÜNLERİ SANAY...,Aktif,İstasyonlu,/Date(1764709199000)/,192.168.1.50;20.1.30.234;192.168.1.140;,37.702188,32.571630,"Istasyon Mahallesi ( Ada:- , Pafta:1 , Parsel:2 )",KONYA,MERAM,"[{'Key': 'AdBlueVar', 'Value': True}, {'Key': ..."
2,100278,70005,1517,BAY/939-82/35867,SAK KAYA PETROL GIDA İNŞAAT SANAYİ TİCARET LİM...,Aktif,İstasyonlu,/Date(1764709199000)/,20.1.26.242;192.168.1.120;,37.153100,33.218400,Siyaser Mahallesi Mersin Yolu Caddesi No:96 ( ...,KARAMAN,KARAMAN,"[{'Key': 'AdBlueVar', 'Value': True}, {'Key': ..."


In [10]:
def check_lpg(features):
    if not isinstance(features, list):
        return False
    for f in features:
        if f.get("Key") == "LPGSatisVar":
            # DEĞİŞİKLIK: == True kısmını sildik, Ruff'ın istediği temiz yazım şekli budur
            return f.get("Value")
    return False


def check_charge(features):
    if not isinstance(features, list):
        return False
    for f in features:
        if f.get("Key") in ["SarjVar", "ElektrikliSarjVar", "SarjSatisVar"]:
            # DEĞİŞİKLIK: == True kısmını sildik
            return f.get("Value")
    return False


# 2. ŞİMDİ ALPET VERİSİNİ TEMİZLEYELİM
df_alpet_temiz = pd.DataFrame()

# Ortak standart kolon isimlerimiz
df_alpet_temiz["istasyon_adi"] = df_alpet_raw["Name"]
df_alpet_temiz["lat"] = df_alpet_raw["Enlem"]
df_alpet_temiz["lon"] = df_alpet_raw["Boylam"]
df_alpet_temiz["brand"] = "Alpet"

# Özellik listesini (StationFeatures) yukarıda tanımladığımız fonksiyonlarla tarıyoruz
df_alpet_temiz["has_lpg"] = df_alpet_raw["StationFeatures"].apply(check_lpg)
df_alpet_temiz["has_charge"] = df_alpet_raw["StationFeatures"].apply(check_charge)

# Koordinatları boş (NaN) olan istasyonları temizleyelim
df_alpet_temiz = df_alpet_temiz.dropna(subset=["lat", "lon"])

# Sonucu ekranda görelim!
df_alpet_temiz.head(5)

,istasyon_adi,lat,lon,brand,has_lpg,has_charge
0,ESKİŞEHİR MERKEZ PETROL VE TİCARET ANONİM ŞİRKETİ,39.797451,30.521778,Alpet,True,False
1,BEYDAĞI PETROL MADENCİLİK TARIM ÜRÜNLERİ SANAY...,37.702188,32.571630,Alpet,True,False
2,SAK KAYA PETROL GIDA İNŞAAT SANAYİ TİCARET LİM...,37.153100,33.218400,Alpet,True,False
3,ALPET İSTASYON İŞLETMELERİ ANONİM ŞİRKETİ,40.860300,29.386400,Alpet,True,False
4,KONYA TURİZM OTO NAKLİYAT VE PETROL ÜRÜNLERİ S...,37.868600,32.536500,Alpet,True,False


In [12]:


# Aytemiz JSON dosyasını okuyalım
with open("../data/raw/benzinistasyonu/aytemiz.json", "r", encoding="utf-8") as f:
    aytemiz_data = json.load(f)

# Pandas DataFrame haline getirip istasyon sayısını kontrol edelim
df_aytemiz_raw = pd.DataFrame(aytemiz_data)
print(f"Aytemiz toplam istasyon sayısı: {len(df_aytemiz_raw)}")

# İlk 3 satırı görerek yapısını teyit edelim
df_aytemiz_raw.head(3)

Aytemiz toplam istasyon sayısı: 800


,City,County,Title,Address,Etiket,Lat,Lon,Phone,Fax,ayt,...,isbank,aytemizkart,motorcudost,castrol,wifi,sarj,selfServis,goon,illykahve,vaay
0,Adana,Yüreğir,As-Ya Petrol Ür. Yemekçilik Otomoti Personel T...,Atatürk Caddesi Havutlu Beld No:580 Yüreğir A...,None,36.931053,35.354591,3223366707,3223363766,VAR,...,YOK,VAR,YOK,VAR,YOK,YOK,YOK,YOK,YOK,VAR
1,Adana,Yüreğir,Öz Ulusal Petrol Nak.Tur.ve Tarım Ü Tic.Ltd.Şti.,Kozan Yolu Ptt Evleri Kavşağ No:693 Yüreğir A...,None,37.011070,35.377133,,,VAR,...,YOK,VAR,YOK,VAR,YOK,YOK,YOK,YOK,YOK,VAR
2,Adana,Sarıçam,Melis Gül (2) Petrol Ürünleri Turiz Taş.İnş.Ot...,Yıldırım Beyazıt Mahalles No: 968/B Sarıçam A...,None,37.026468,35.401303,,,VAR,...,YOK,VAR,YOK,VAR,YOK,YOK,YOK,YOK,YOK,VAR


In [13]:
# Sadece projemiz için ortak belirlediğimiz kolonları seçelim
# Orijinal kolonlar: Title (Adı), Lat (Enlem), Lon (Boylam), sarj (Şarj), gaz (LPG)
df_aytemiz_temiz = df_aytemiz_raw[["Title", "Lat", "Lon", "sarj", "gaz"]].copy()

# Kolon isimlerini ortaklaştıralım
df_aytemiz_temiz.columns = ["istasyon_adi", "lat", "lon", "has_charge", "has_lpg"]

# "Lat" ve "Lon" metinsel (string) değerlerini sayısal (float) değerlere dönüştürelim
# Hatalı/boş girilen değerler varsa NaN (boş değer) yapması için errors="coerce" kullanıyoruz
df_aytemiz_temiz["lat"] = pd.to_numeric(df_aytemiz_temiz["lat"], errors="coerce")
df_aytemiz_temiz["lon"] = pd.to_numeric(df_aytemiz_temiz["lon"], errors="coerce")

# "VAR" -> True, "YOK" -> False olacak şekilde mantıksal tipe (bool) çevirelim
df_aytemiz_temiz["has_charge"] = df_aytemiz_temiz["has_charge"] == "VAR"
df_aytemiz_temiz["has_lpg"] = df_aytemiz_temiz["has_lpg"] == "VAR"

# Marka bilgisini "Aytemiz" olarak ekleyelim
df_aytemiz_temiz["brand"] = "Aytemiz"

# Koordinatları eksik/hatalı olan satırları temizleyelim
df_aytemiz_temiz = df_aytemiz_temiz.dropna(subset=["lat", "lon"])

# Temizlenmiş Aytemiz tablosuna göz atalım!
df_aytemiz_temiz.head(5)

,istasyon_adi,lat,lon,has_charge,has_lpg,brand
0,As-Ya Petrol Ür. Yemekçilik Otomoti Personel T...,36.931053,35.354591,False,True,Aytemiz
1,Öz Ulusal Petrol Nak.Tur.ve Tarım Ü Tic.Ltd.Şti.,37.011070,35.377133,False,True,Aytemiz
2,Melis Gül (2) Petrol Ürünleri Turiz Taş.İnş.Ot...,37.026468,35.401303,False,True,Aytemiz
3,Kan Akaryakıt Petrol Ürünleri Otomo İnşaat Gıd...,37.029884,35.817420,False,True,Aytemiz
4,Serhat Zirai Ürünler Ticari Limited Şirketi,37.017123,35.798464,True,True,Aytemiz


In [15]:
# Kadoil JSON dosyasını okuyalım
# (Eğer kadoil.json dosyasının adı farklıysa burayı ona göre güncelleyebilirsin)
with open("../data/raw/benzinistasyonu/kadoil.json", "r", encoding="utf-8") as f:
    kadoil_data = json.load(f)

# Pandas DataFrame haline getirip istasyon sayısına bakalım
df_kadoil_raw = pd.DataFrame(kadoil_data)
print(f"Kadoil toplam istasyon sayısı: {len(df_kadoil_raw)}")

# İlk 3 satırını görerek yapıyı teyit edelim
df_kadoil_raw.head(3)

Kadoil toplam istasyon sayısı: 519


,province,district,name,address,lat,lng,services,phone
0,adana,Şehir Merkezi,Geylaniler Petrol Ürünleri Ticaret Ve Sanayi L...,Cumhuriyet Caddesi 27. sokak Sokağı No:34 ( Ad...,36.965760,35.609555,[KADOMATİK],NaN
1,adana,YÜREĞİR,ALTINYÜZÜK GIDA AKARYAKIT VE PETROL ÜRÜNLERİ İ...,Taşçı Mahallesi Karataş Bulvarı No:925/A ( Ada...,36.873306,35.346085,NaN,NaN
2,adana,KOZAN,Tanrıverdi Petrol Nakliyat Gıda Turizm Sanayi ...,Sırkıntı Caddesi No:342 Şevkiye Mahallesi Koza...,37.442945,35.780642,[KADOMATİK],NaN


In [17]:
# 1. Kadoil için özellik kontrol fonksiyonlarını yazalım
def check_kadoil_lpg(services):
    # Eğer services alanı yoksa (float/NaN ise) veya liste değilse direkt False dön
    if not isinstance(services, list):
        return False
    # Liste içindeki servisleri küçük harfe çevirerek "lpg" araması yapalım
    for s in services:
        if "lpg" in s.lower() or "otogaz" in s.lower():
            return True
    return False

def check_kadoil_charge(services):
    if not isinstance(services, list):
        return False
    # Liste içinde "şarj", "sarj", "ev" veya "elektrik" kelimelerini arayalım
    for s in services:
        s_lower = s.lower()
        if "şarj" in s_lower or "sarj" in s_lower or "ev" in s_lower or "elektrik" in s_lower:
            return True
    return False


# 2. Temiz DataFrame'i oluşturalım
df_kadoil_temiz = pd.DataFrame()

# Ortak standart sütun isimlerimizi eşleştiriyoruz
df_kadoil_temiz["istasyon_adi"] = df_kadoil_raw["name"]
df_kadoil_temiz["lat"] = df_kadoil_raw["lat"]
df_kadoil_temiz["lon"] = df_kadoil_raw["lng"] # lng -> lon dönüştürmesi
df_kadoil_temiz["brand"] = "Kadoil"

# Eğer ham veride "services" sütunu hiç yoksa hata vermemesi için kontrol ekliyoruz
if "services" in df_kadoil_raw.columns:
    df_kadoil_temiz["has_lpg"] = df_kadoil_raw["services"].apply(check_kadoil_lpg)
    df_kadoil_temiz["has_charge"] = df_kadoil_raw["services"].apply(check_kadoil_charge)
else:
    # "services" sütunu hiç yoksa tümünü varsayılan olarak False yapıyoruz
    df_kadoil_temiz["has_lpg"] = False
    df_kadoil_temiz["has_charge"] = False

# Koordinatları boş (NaN) olan istasyonları temizleyelim
df_kadoil_temiz = df_kadoil_temiz.dropna(subset=["lat", "lon"])

# Temizlenmiş Kadoil tablosuna bir bakalım!
df_kadoil_temiz.head(5)

,istasyon_adi,lat,lon,brand,has_lpg,has_charge
0,Geylaniler Petrol Ürünleri Ticaret Ve Sanayi L...,36.965760,35.609555,Kadoil,False,False
1,ALTINYÜZÜK GIDA AKARYAKIT VE PETROL ÜRÜNLERİ İ...,36.873306,35.346085,Kadoil,False,False
2,Tanrıverdi Petrol Nakliyat Gıda Turizm Sanayi ...,37.442945,35.780642,Kadoil,False,False
3,Mobay Petrol Turizm Tarım Gübre İnşaat Gıda Fı...,37.491919,35.831750,Kadoil,False,False
4,İZ PETROL İNŞAAT NAKLİYAT GIDA GÜBRE MADEN İTH...,37.424777,35.788907,Kadoil,False,False


In [19]:
# Opet JSON dosyasını okuyalım
# (Eğer opet.json dosyasının adı farklıysa burayı ona göre güncelleyebilirsin)
with open("../data/raw/benzinistasyonu/opet.json", "r", encoding="utf-8") as f:
    opet_data = json.load(f)

# Pandas DataFrame haline getirip istasyon sayısına bakalım
df_opet_raw = pd.DataFrame(opet_data)
print(f"Opet toplam istasyon sayısı: {len(df_opet_raw)}")

# İlk 3 satırını görerek yapıyı teyit edelim
df_opet_raw.head(3)

Opet toplam istasyon sayısı: 1228


,id,name,address,province,district,phone,fax,longitude,latitude,vkgrp,kunnr,email,kep,taxNo,taxOffice,featureCategories
0,9000000670,KALAYOĞLU PETROL ÜRÜNLERİ KUY. NAK.TUR.İNŞ. S...,ASİLBEY MAH. YAŞAR ÇELİK CAD. NO:82 İÇ KAPI NO:A,BOLU,YENİÇAĞA,5334310464,,32.04025,40.77374,114,0000304360,,,4900032950,DEVREK,"[{'id': 12, 'label': 'Ulusal Taşıt Tanıma Sist..."
1,9000000676,GÖREN PETROL TESİSLERİ SANAYİ LİMİTED ŞİRKETİ,DEĞİRMENÖNÜ MAH. ANKARA YOLU C AD. NO:800A,BURSA,YILDIRIM,2243319731,2243319072,29.17320,40.19721,122,0000301549,,gorenpetrol@hs01.kep.tr,4090033416,GÖKDERE,"[{'id': 12, 'label': 'Ulusal Taşıt Tanıma Sist..."
2,9000001405,HURİYE GÖK,SELÇUK MAH. HANÇERLİ MEVKİİ,BURSA,İZNİK,2247570192,2247575885,29.71669,40.41909,121,0000300139,,huriye.gok@hs01.kep.tr,30850254568,İZNİK,"[{'id': 12, 'label': 'Ulusal Taşıt Tanıma Sist..."


In [21]:
# 1. Opet'in iç içe geçmiş özellik listelerini tarayan fonksiyonları yazalım
def check_opet_lpg(categories):
    # Eğer kategori alanı yoksa veya liste değilse direkt False dön
    if not isinstance(categories, list):
        return False
    
    # Her bir kategorinin içine tek tek giriyoruz
    for category in categories:
        features = category.get("stationFeatures", [])
        # Kategorinin altındaki özelliklerin içine giriyoruz
        for f in features:
            label = f.get("label", "")
            if label and ("lpg" in label.lower() or "otogaz" in label.lower()):
                return True
    return False

def check_opet_charge(categories):
    if not isinstance(categories, list):
        return False
    
    for category in categories:
        features = category.get("stationFeatures", [])
        for f in features:
            label = f.get("label", "")
            if label:
                label_lower = label.lower()
                # Şarj, sarj, elektrikli veya ev kelimelerinden birini arıyoruz
                if "şarj" in label_lower or "sarj" in label_lower or "elektrik" in label_lower or "ev " in label_lower:
                    return True
    return False


# 2. Temiz DataFrame'i oluşturalım
df_opet_temiz = pd.DataFrame()

# Ortak standart sütun isimlerimizi Opet'e göre eşleştiriyoruz
df_opet_temiz["istasyon_adi"] = df_opet_raw["name"]
df_opet_temiz["lat"] = df_opet_raw["latitude"]
df_opet_temiz["lon"] = df_opet_raw["longitude"]
df_opet_temiz["brand"] = "Opet"

# Özellikleri tarayıp True/False olarak ekleyelim
if "featureCategories" in df_opet_raw.columns:
    df_opet_temiz["has_lpg"] = df_opet_raw["featureCategories"].apply(check_opet_lpg)
    df_opet_temiz["has_charge"] = df_opet_raw["featureCategories"].apply(check_opet_charge)
else:
    df_opet_temiz["has_lpg"] = False
    df_opet_temiz["has_charge"] = False

# Koordinatları boş (NaN) olan istasyonları temizleyelim
df_opet_temiz = df_opet_temiz.dropna(subset=["lat", "lon"])

# Temizlenmiş Opet tablosunun ilk 5 satırına bakalım!
df_opet_temiz.head(5)

,istasyon_adi,lat,lon,brand,has_lpg,has_charge
0,KALAYOĞLU PETROL ÜRÜNLERİ KUY. NAK.TUR.İNŞ. S...,40.773740,32.040250,Opet,True,False
1,GÖREN PETROL TESİSLERİ SANAYİ LİMİTED ŞİRKETİ,40.197210,29.173200,Opet,True,False
2,HURİYE GÖK,40.419090,29.716690,Opet,True,False
3,ASIM VE İKİZOĞULLARI PET.ÜRÜN. GIDA İNŞ.SA.TİC...,41.008389,29.216246,Opet,True,False
4,ÖZTÜRKLER TARIM VE PETROL ÜRÜNLERİ TİCARET LİM...,40.777150,29.712280,Opet,True,False


In [25]:
# Petrol Ofisi JSON dosyasını okuyalım
with open("../data/raw/benzinistasyonu/petrol_ofisi.json", "r", encoding="utf-8") as f:
    po_raw_data = json.load(f)

# İç içe geçmiş "Values" listelerini tek bir düz listede toplayalım
flat_po_stations = []

# Eğer veri doğrudan bir liste ise ve içinde şehir objeleri varsa
for city_data in po_raw_data:
    if isinstance(city_data, dict) and "Values" in city_data:
        # Şehir altındaki istasyonları listemize ekliyoruz
        flat_po_stations.extend(city_data["Values"])

# Düzleştirilmiş veriyi DataFrame yapalım
df_po_raw = pd.DataFrame(flat_po_stations)
print(f"Petrol Ofisi toplam istasyon sayısı: {len(df_po_raw)}")

# İlk 3 satırı kontrol edelim
df_po_raw.head(3)

Petrol Ofisi toplam istasyon sayısı: 2672


,Id,StationName,Address,CityName,DistrictName,PhoneNumber,Latitude,Longitude,CityId,DistrictId,Zone,Services,Url
0,1699,1. MURAT PETROL OFISI,"1.MURAT MAHALLESİ TALATPAŞA CAD.NO:64, MERKEZ,...",Edirne,MERKEZ,02842361222,41.662883,26.578193,22,02200,Batı Marmara,10010001010000110001101011000010101,1-murat-petrol-ofisi-1699
1,1018,ADASARHANLI KÖYÜ PETROL OFISI,"ADASARHANLI KÖYÜ, MERİÇ, Edirne",Edirne,MERİÇ,02844391190,41.083913,26.355788,22,02206,Batı Marmara,00010000000000000000000000000000101,adasarhanli-koyu-petrol-ofisi-1018
2,136,ADASARHANLI MERIÇ YOLU PETROL OFISI,"ADASARHANLI KÖYÜ, MERİÇ, Edirne",Edirne,MERİÇ,02844391015,41.086027,26.349995,22,02206,Batı Marmara,00000000000000000000000000000000100,adasarhanli-meric-yolu-petrol-ofisi-136


In [27]:
# 1. Petrol Ofisi için özellik kontrol fonksiyonlarını yazalım
def check_po_lpg(row):
    # İstasyon adı veya adresinde LPG araması yapalım
    name = str(row.get("StationName", "")).lower()
    address = str(row.get("Address", "")).lower()
    
    if "lpg" in name or "otogaz" in name or "po/gaz" in name or "pogaz" in name:
        return True
    if "lpg" in address or "otogaz" in address:
        return True
        
    # DEĞİŞİKLİK: Kullanılmayan services satırını tamamen sildik.
    return False

def check_po_charge(row):
    # İstasyon adı veya adresinde Şarj araması yapalım
    name = str(row.get("StationName", "")).lower()
    address = str(row.get("Address", "")).lower()
    
    # Petrol Ofisi'nin kendi şarj markası "e-POwer"dır.
    if "şarj" in name or "sarj" in name or "epower" in name or "e-power" in name:
        return True
    if "şarj" in address or "sarj" in address or "elektrikli şarj" in address:
        return True
        
    return False


# 2. Temiz DataFrame'i oluşturalım
df_po_temiz = pd.DataFrame()

# Standart ortak sütunlarımızı eşleştiriyoruz
df_po_temiz["istasyon_adi"] = df_po_raw["StationName"]
df_po_temiz["lat"] = df_po_raw["Latitude"]
df_po_temiz["lon"] = df_po_raw["Longitude"]
df_po_temiz["brand"] = "Petrol Ofisi"

# Fonksiyonlarımızı tüm satırlara (axis=1) uygulayarak özellikleri bulalım
df_po_temiz["has_lpg"] = df_po_raw.apply(check_po_lpg, axis=1)
df_po_temiz["has_charge"] = df_po_raw.apply(check_po_charge, axis=1)

# Koordinatları boş (NaN) veya 0 olan istasyonları temizleyelim
df_po_temiz = df_po_temiz.dropna(subset=["lat", "lon"])
df_po_temiz = df_po_temiz[(df_po_temiz["lat"] != 0) & (df_po_temiz["lon"] != 0)]

# Temizlenmiş Petrol Ofisi tablosuna bir bakalım!
df_po_temiz.head(5)

,istasyon_adi,lat,lon,brand,has_lpg,has_charge
0,1. MURAT PETROL OFISI,41.662883,26.578193,Petrol Ofisi,False,False
1,ADASARHANLI KÖYÜ PETROL OFISI,41.083913,26.355788,Petrol Ofisi,False,False
2,ADASARHANLI MERIÇ YOLU PETROL OFISI,41.086027,26.349995,Petrol Ofisi,False,False
3,BOSTANPAZARI CAD. PETROL OFİSİ,41.667306,26.558651,Petrol Ofisi,False,False
4,ÇÖPKÖY PETROL OFISI,41.225952,26.810944,Petrol Ofisi,False,False


In [29]:
# Shell JSON dosyasını okuyalım
with open("../data/raw/benzinistasyonu/shell.json", "r", encoding="utf-8") as f:
    shell_data = json.load(f)

# Pandas DataFrame haline getirip istasyon sayısına bakalım
df_shell_raw = pd.DataFrame(shell_data)
print(f"Shell toplam istasyon sayısı: {len(df_shell_raw)}")

# İlk 3 satırını görerek yapıyı teyit edelim
df_shell_raw.head(3)

Shell toplam istasyon sayısı: 1416


,id,name,lat,lng,brand,inactive,type,country_code,address,city,...,open_status,next_open_status_change,tz_offset,fuels,next_forecourt_open_status_change,next_shop_open_status_change,next_ev_open_status_change,forecourt_open_status,shop_open_status,ev_open_status
0,10095815,ΕΡΜΗΣ ΑΕΜΕΕ ΥΠ/ΜΑ ΣΑΝΤΟΡΙΝΗΣ,36.409937,25.457943,Shell,False,0,GR,"ΝΕΑ ΟΔΟΣ ΑΕΡΟΔΡΟΜΙΟΥ – ΦΗΡΩΝ – ΒΟΥΡΒΟΥΛΩΝ, ΘΕΣ...",ΣΑΝΤΟΡΙΝΗ,...,open,2025-12-02T20:00:00.546Z,7200,"[premium_diesel, fuelsave_regular_diesel, midg...",2025-12-02T20:00:00.550Z,NaN,NaN,open,unknown,unknown
1,10095213,ΦΟΥΣΤΕΡΗΣ Σ. & ΣΙΑ ΟΕ,36.407908,25.445795,Shell,False,0,GR,ΚΑΡΤΕΡΑΔΟΣ,ΘΗΡΑ,...,open,2025-12-02T19:00:00.563Z,7200,"[premium_diesel, fuelsave_regular_diesel, midg...",2025-12-02T19:00:00.566Z,NaN,NaN,open,unknown,unknown
2,10095523,ΝΑΞΟΣ ΑΕ,35.245439,25.725123,Shell,False,0,GR,2 ΧΛΜ ΕΠΑΡΧΙΑΚΗΣ ΟΔΟΥ ΕΛΟΥΝΤΑΣ - ΑΓ. ΝΙΚΟΛΑΟΥ,ΑΓ. ΝΙΚΟΛΑΟΣ,...,open,2025-12-02T19:00:00.580Z,7200,"[premium_diesel, fuelsave_regular_diesel, midg...",2025-12-02T19:00:00.583Z,NaN,NaN,open,unknown,unknown


In [31]:
# 1. Shell için özellik kontrol fonksiyonlarını yazalım
def check_shell_lpg(fuels_list):
    if not isinstance(fuels_list, list):
        return False
    # "autogas_lpg" veya "lpg" kelimesini yakıt listesinde arayalım
    for f in fuels_list:
        if "lpg" in str(f).lower():
            return True
    return False

def check_shell_charge(amenities_list):
    if not isinstance(amenities_list, list):
        return False
    # Hizmetler listesinde "ev_", "charge", "şarj", "sarj" kelimelerini arayalım
    for a in amenities_list:
        a_lower = str(a).lower()
        if "ev_" in a_lower or "charge" in a_lower or "şarj" in a_lower or "sarj" in a_lower:
            return True
    return False


# 2. Temiz DataFrame'i oluşturalım
df_shell_temiz = pd.DataFrame()

# Eğer veride farklı ülkeler varsa sadece Türkiye'yi ("TR") seçelim
# (Eğer "country_code" kolonu yoksa hatayı önlemek için doğrudan ham veriyi kullanalım)
if "country_code" in df_shell_raw.columns:
    df_shell_filtered = df_shell_raw[df_shell_raw["country_code"] == "TR"].copy()
else:
    df_shell_filtered = df_shell_raw.copy()

# Standart ortak sütunlarımızı eşleştiriyoruz
df_shell_temiz["istasyon_adi"] = df_shell_filtered["name"]
df_shell_temiz["lat"] = df_shell_filtered["lat"]
df_shell_temiz["lon"] = df_shell_filtered["lng"] # lng -> lon dönüştürmesi
df_shell_temiz["brand"] = "Shell"

# Özellik listelerini tarayıp True/False olarak ekleyelim
if "fuels" in df_shell_filtered.columns:
    df_shell_temiz["has_lpg"] = df_shell_filtered["fuels"].apply(check_shell_lpg)
else:
    df_shell_temiz["has_lpg"] = False

if "amenities" in df_shell_filtered.columns:
    df_shell_temiz["has_charge"] = df_shell_filtered["amenities"].apply(check_shell_charge)
else:
    df_shell_temiz["has_charge"] = False

# Koordinatları boş (NaN) olan istasyonları temizleyelim
df_shell_temiz = df_shell_temiz.dropna(subset=["lat", "lon"])

# Temizlenmiş Shell tablosuna bir bakalım!
df_shell_temiz.head(5)

,istasyon_adi,lat,lon,brand,has_lpg,has_charge
51,TURGUT REİS BODRUM.,37.010007,27.267088,Shell,True,False
52,DATÇA,36.763699,27.723933,Shell,True,False
53,ACIBADEM H-BODRM,37.055020,27.348750,Shell,False,True
54,ORTAKENT BODRUM,37.055898,27.350957,Shell,True,False
56,KONACIK.,37.049100,27.391251,Shell,True,True


In [34]:
# Soil JSON dosyasını okuyalım
with open("../data/raw/benzinistasyonu/soil_stations.json", "r", encoding="utf-8") as f:
    soil_data = json.load(f)

# Pandas DataFrame haline getirip istasyon sayısına bakalım
df_soil_raw = pd.DataFrame(soil_data)
print(f"Soil toplam istasyon sayısı: {len(df_soil_raw)}")

# İlk 3 satırını görerek yapıyı teyit edelim
df_soil_raw.head(3)

Soil toplam istasyon sayısı: 347


,id,title,description,street,city,state,postal_code,country,lat,lng,...,path,marker_id,description_2,open_hours,ordr,slug,brand,special,categories,days_str
0,67,33 LOJİSTİK PETROL,BAY/93982/33200,Yarbay Şemsettin Mahallesi Mahallesi Ankara Bu...,Tarsus,Mersin(İçel),,Turkey,36.93788179999999,34.932734682031196,...,soil-logo.png,151,,"{""mon"":""0"",""tue"":""0"",""wed"":""0"",""thu"":""0"",""fri""...",0,33-loj-st-k-petrol-tarsus,,,19,
1,133,A.ÖZTÜRK PETROL,BAY/463-1088/09723,Şanlıurfa Karayolu 3. Km.,Viranşehir,Şanlıurfa,,Turkey,37.234773644665296,39.76272132275392,...,soil-logo.png,151,,"{""mon"":""0"",""tue"":""0"",""wed"":""0"",""thu"":""0"",""fri""...",0,a-zt-rk-petrol-viran-ehir,,,19,
2,158,ACAR PETROL,BAY/939-82/25601,Yeni Mahalle Cizre Karayolu 10. Km,Merkez,Şırnak,,Turkey,37.51746138795097,42.45179170000006,...,soil-logo.png,151,,"{""mon"":""0"",""tue"":""0"",""wed"":""0"",""thu"":""0"",""fri""...",0,acar-petrol-merkez,,,19,


In [36]:
# 1. Soil için istasyon adından özellik yakalama fonksiyonlarını yazalım
def check_soil_lpg(title):
    title_lower = str(title).lower()
    if "lpg" in title_lower or "otogaz" in title_lower or "gaz" in title_lower:
        return True
    return False

def check_soil_charge(title):
    title_lower = str(title).lower()
    if "şarj" in title_lower or "sarj" in title_lower or "ev " in title_lower:
        return True
    return False


# 2. Temiz DataFrame'i oluşturalım
df_soil_temiz = pd.DataFrame()

# Standart ortak sütunlarımızı eşleştiriyoruz
df_soil_temiz["istasyon_adi"] = df_soil_raw["title"]

# Metinsel koordinatları sayısal (float) tipe dönüştürüyoruz
df_soil_temiz["lat"] = pd.to_numeric(df_soil_raw["lat"], errors="coerce")
df_soil_temiz["lon"] = pd.to_numeric(df_soil_raw["lng"], errors="coerce") # lng -> lon dönüştürmesi

df_soil_temiz["brand"] = "Soil"

# Özellikleri istasyon adına göre tarayıp ekleyelim
df_soil_temiz["has_lpg"] = df_soil_raw["title"].apply(check_soil_lpg)
df_soil_temiz["has_charge"] = df_soil_raw["title"].apply(check_soil_charge)

# Koordinatları boş (NaN) veya hatalı olan satırları temizleyelim
df_soil_temiz = df_soil_temiz.dropna(subset=["lat", "lon"])

# Temizlenmiş Soil tablosunun ilk 5 satırına göz atalım!
df_soil_temiz.head(5)

,istasyon_adi,lat,lon,brand,has_lpg,has_charge
0,33 LOJİSTİK PETROL,36.937882,34.932735,Soil,False,False
1,A.ÖZTÜRK PETROL,37.234774,39.762721,Soil,False,False
2,ACAR PETROL,37.517461,42.451792,Soil,False,False
3,ACUN KARDEŞLER AKARYAKIT,38.245476,34.132245,Soil,False,False
4,ADAPET PETROL,40.796706,29.441033,Soil,False,False


In [37]:
# TotalEnergies JSON dosyasını okuyalım
with open("../data/raw/benzinistasyonu/total_stations.json", "r", encoding="utf-8") as f:
    total_data = json.load(f)

# Pandas DataFrame haline getirip istasyon sayısına bakalım
df_total_raw = pd.DataFrame(total_data)
print(f"TotalEnergies toplam istasyon sayısı: {len(df_total_raw)}")

# İlk 3 satırını görerek yapıyı teyit edelim
df_total_raw.head(3)

TotalEnergies toplam istasyon sayısı: 762


,id,name,lat,lon,icon,info,cat1,cat2,cat3,cat4,cat5,city,cityName,county,countyName,district,districtName,data
0,Total_Z000652,TotalEnergies FINDIKLI,41.280752,41.153746,/static/poi_icons/TotalEnergies.png,{'address': 'Aksu Mahallesi 21. Caddesi No:28 ...,TotalEnergies,TotalEnergies,TotalEnergies,TotalEnergies,TotalEnergies,53,Rize,1332,Fındıklı,61404,Aksu Mah,"{'wash': None, 'yakitmatik': 'VAR', 'elle_yika..."
1,Total_Z01G362,TotalEnergies İNCİRLİK,36.974639,35.490972,/static/poi_icons/TotalEnergies.png,{'address': 'Yakapınar Mahallesi D-400 Bulvarı...,TotalEnergies,TotalEnergies,TotalEnergies,TotalEnergies,TotalEnergies,1,Adana,1748,Yüreğir,223,Yakapınar Mah,"{'wash': None, 'yakitmatik': None, 'elle_yikam..."
2,Total_Z07G325,TotalEnergies GÜVERCİNLİK,36.880450,31.256070,/static/poi_icons/TotalEnergies.png,{'address': 'Hocalar Mahallesi Aşağı Kümeevler...,TotalEnergies,TotalEnergies,TotalEnergies,TotalEnergies,TotalEnergies,7,Antalya,1512,Manavgat,181989,Hocalar Mah,"{'wash': None, 'yakitmatik': None, 'elle_yikam..."


In [38]:
# 1. TotalEnergies için özellik kontrol fonksiyonlarını yazalım
def check_total_lpg(name):
    name_lower = str(name).lower()
    if "lpg" in name_lower or "otogaz" in name_lower or "gaz" in name_lower:
        return True
    return False

def check_total_charge(name):
    name_lower = str(name).lower()
    if "şarj" in name_lower or "sarj" in name_lower or "charge" in name_lower or "ev " in name_lower:
        return True
    return False


# 2. Temiz DataFrame'i oluşturalım
df_total_temiz = pd.DataFrame()

# Standart ortak sütunlarımızı eşleştiriyoruz
df_total_temiz["istasyon_adi"] = df_total_raw["name"]
df_total_temiz["lat"] = df_total_raw["lat"]
df_total_temiz["lon"] = df_total_raw["lon"]
df_total_temiz["brand"] = "TotalEnergies"

# Özellikleri istasyon adına göre tarayıp True/False olarak ekleyelim
df_total_temiz["has_lpg"] = df_total_raw["name"].apply(check_total_lpg)
df_total_temiz["has_charge"] = df_total_raw["name"].apply(check_total_charge)

# Koordinatları boş (NaN) olan istasyonları temizleyelim
df_total_temiz = df_total_temiz.dropna(subset=["lat", "lon"])

# Temizlenmiş TotalEnergies tablosuna göz atalım!
df_total_temiz.head(5)

,istasyon_adi,lat,lon,brand,has_lpg,has_charge
0,TotalEnergies FINDIKLI,41.280752,41.153746,TotalEnergies,False,False
1,TotalEnergies İNCİRLİK,36.974639,35.490972,TotalEnergies,False,False
2,TotalEnergies GÜVERCİNLİK,36.880450,31.256070,TotalEnergies,False,False
3,TotalEnergies ABANT,40.718460,31.470800,TotalEnergies,False,False
4,TotalEnergies KIBYRA,37.288320,29.554730,TotalEnergies,False,False


In [39]:
# Türkiye Petrolleri JSON dosyasını okuyalım
with open("../data/raw/benzinistasyonu/turkiye_petrolleri.json", "r", encoding="utf-8") as f:
    tp_data = json.load(f)

# Pandas DataFrame haline getirip istasyon sayısına bakalım
df_tp_raw = pd.DataFrame(tp_data)
print(f"Türkiye Petrolleri toplam istasyon sayısı: {len(df_tp_raw)}")

# İlk 3 satırını görerek yapıyı teyit edelim
df_tp_raw.head(3)

Türkiye Petrolleri toplam istasyon sayısı: 107


,StationName,Lat,Lng,Address,County,CityName,CityId,StationType,VIS,KursunsuzBenzin,TpMotorin,TpGaz
0,ABDULKADİR EROĞLU PETROL,37.73410,38.23540,Altınşehir Mahallesi Gazihan Caddesi No:56 MER...,MERKEZ,ADIYAMAN,2,1,True,0,0,0
1,ADA ARİF PETROL,40.74090,30.42770,Kozluk Mahallesi D-100 Yanyol Caddesi No:193 E...,ERENLER,SAKARYA,54,1,True,0,0,0
2,ADEM ÖZDEMİR PETROL,38.33327,28.56159,ILICA MAH. DENİZLİ YOLU CAD.NO 202 ALAŞEHİR MA...,ALAŞEHİR,MANİSA,45,1,True,0,0,0


In [40]:
# 1. Türkiye Petrolleri için özellik kontrol fonksiyonlarını yazalım
def check_tp_lpg(row):
    # Eğer TpGaz alanı 0'dan büyükse veya boş değilse LPG var kabul edebiliriz
    tp_gaz = row.get("TpGaz")
    if tp_gaz and tp_gaz != 0:
        return True
        
    # İstasyon adından da kontrol edelim (yedek plan)
    name_lower = str(row.get("StationName", "")).lower()
    if "lpg" in name_lower or "otogaz" in name_lower or "gaz" in name_lower:
        return True
    return False

def check_tp_charge(row):
    # İstasyon adında Şarj araması yapalım
    name_lower = str(row.get("StationName", "")).lower()
    if "şarj" in name_lower or "sarj" in name_lower or "charge" in name_lower or "ev " in name_lower:
        return True
    return False


# 2. Temiz DataFrame'i oluşturalım
df_tp_temiz = pd.DataFrame()

# Standart ortak sütunlarımızı eşleştiriyoruz
df_tp_temiz["istasyon_adi"] = df_tp_raw["StationName"]
df_tp_temiz["lat"] = df_tp_raw["Lat"]
df_tp_temiz["lon"] = df_tp_raw["Lng"] # Lng -> lon dönüştürmesi
df_tp_temiz["brand"] = "Türkiye Petrolleri"

# Satır bazlı (axis=1) özellik taraması yapalım
df_tp_temiz["has_lpg"] = df_tp_raw.apply(check_tp_lpg, axis=1)
df_tp_temiz["has_charge"] = df_tp_raw.apply(check_tp_charge, axis=1)

# Koordinatları boş (NaN) olan istasyonları temizleyelim
df_tp_temiz = df_tp_temiz.dropna(subset=["lat", "lon"])

# Temizlenmiş Türkiye Petrolleri tablosuna göz atalım!
df_tp_temiz.head(5)

,istasyon_adi,lat,lon,brand,has_lpg,has_charge
0,ABDULKADİR EROĞLU PETROL,37.734100,38.23540,Türkiye Petrolleri,False,False
1,ADA ARİF PETROL,40.740900,30.42770,Türkiye Petrolleri,False,False
2,ADEM ÖZDEMİR PETROL,38.333270,28.56159,Türkiye Petrolleri,False,False
3,Adnan Erdoğmuş,39.504013,34.75717,Türkiye Petrolleri,False,False
4,ALPTAN PETROL,38.777900,30.49830,Türkiye Petrolleri,False,False


In [42]:
import os
import pandas as pd
# 1. Temizlediğimiz tüm DataFrame'leri bir liste içinde toplayalım
tum_temiz_tablolar = [
    df_alpet_temiz,
    df_aytemiz_temiz,
    df_kadoil_temiz,
    df_opet_temiz,
    df_po_temiz,
    df_shell_temiz,
    df_soil_temiz,
    df_total_temiz,
    df_tp_temiz
]

# 2. Hepsini alt alta yapıştırarak master (ana) veri setini oluşturalım
# ignore_index=True satır numaralarını sıfırlayıp ardışık (0, 1, 2...) yapar.
df_harita_master = pd.concat(tum_temiz_tablolar, ignore_index=True)

# 3. Klasör yapımızı kontrol edip oluşturalım ve veriyi kaydedelim
output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "temiz_istasyon_verileri.csv")

# Veriyi CSV formatında kaydediyoruz
df_harita_master.to_csv(output_path, index=False, encoding="utf-8")

# 4. Genel İstatistikleri Ekrana Basalım
print("=============================================")
print(" 🎉 DEV VERİ KÜMESİ BAŞARIYLA OLUŞTURULDU! 🎉")
print("=============================================")
print(f"Haritaya Basılacak Toplam İstasyon Sayısı: {len(df_harita_master)}")
print(f"Kayıt Yeri: {output_path}")
print("\nMarkalara Göre İstasyon Dağılımı:")
print(df_harita_master["brand"].value_counts())
print("=============================================\n")

# İlk 5 satıra genel bir bakış atalım
df_harita_master.head(5)

 🎉 DEV VERİ KÜMESİ BAŞARIYLA OLUŞTURULDU! 🎉
Haritaya Basılacak Toplam İstasyon Sayısı: 8024
Kayıt Yeri: ../data/processed\temiz_istasyon_verileri.csv

Markalara Göre İstasyon Dağılımı:
brand
Petrol Ofisi          2672
Shell                 1232
Opet                  1228
Aytemiz                800
TotalEnergies          762
Kadoil                 503
Alpet                  373
Soil                   347
Türkiye Petrolleri     107
Name: count, dtype: int64



,istasyon_adi,lat,lon,brand,has_lpg,has_charge
0,ESKİŞEHİR MERKEZ PETROL VE TİCARET ANONİM ŞİRKETİ,39.797451,30.521778,Alpet,True,False
1,BEYDAĞI PETROL MADENCİLİK TARIM ÜRÜNLERİ SANAY...,37.702188,32.571630,Alpet,True,False
2,SAK KAYA PETROL GIDA İNŞAAT SANAYİ TİCARET LİM...,37.153100,33.218400,Alpet,True,False
3,ALPET İSTASYON İŞLETMELERİ ANONİM ŞİRKETİ,40.860300,29.386400,Alpet,True,False
4,KONYA TURİZM OTO NAKLİYAT VE PETROL ÜRÜNLERİ S...,37.868600,32.536500,Alpet,True,False


In [1]:
import pandas as pd

# Olası tüm şehir sütunu isimlerinin listesi
sehir_sutunlari = ["CityName", "City", "province", "state", "cityName"]


def sehir_sutununu_standartlastir(df):
    for col in df.columns:
        if col in sehir_sutunlari:
            # Farklı adı yakaladığımız an standart olarak 'city' yapıyoruz
            df = df.rename(columns={col: "city"})
            break
    return df

In [2]:
def sehir_ismini_temizle(metin):
    if pd.isna(metin):
        return "Bilinmiyor"

    metin = str(metin).strip().upper()  # Baştaki/sondaki boşlukları sil ve büyüt

    # Türkçe karakter uyumsuzlukları için ufak bir düzeltme
    metin = metin.replace("İ", "I").replace("Ğ", "G").replace("Ü", "U")
    metin = metin.replace("Ş", "S").replace("Ö", "O").replace("Ç", "C")

    return metin


# Tüm veriye tek satırda uygularız:
# df['city'] = df['city'].apply(sehir_ismini_temizle)